## Olist Dataset - Kaggle: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

### Step 1: Setting Up the Spark Environment

In a real-world company setup, we wouldn’t use Google Colab directly. Instead, we would:

* **Deploy a Spark Cluster** (like AWS EMR, GCP Dataproc, or an on-prem Hadoop cluster, Azure HD Insight).
*  **Store Data in HDFS** instead of local storage.
*   Load data from Kaggle
*  **Use PySpark** to interact with data.

In [1]:
#!/bin/bash
!curl -L -o /tmp/brazilian-ecommerce.zip\
  https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 42.6M  100 42.6M    0     0   111M      0 --:--:-- --:--:-- --:--:--  111M


In [2]:
!unzip -o /tmp/brazilian-ecommerce.zip -d /tmp/brazilian-ecommerce

Archive:  /tmp/brazilian-ecommerce.zip
  inflating: /tmp/brazilian-ecommerce/olist_customers_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_geolocation_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_order_items_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_order_payments_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_order_reviews_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_orders_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_products_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/olist_sellers_dataset.csv  
  inflating: /tmp/brazilian-ecommerce/product_category_name_translation.csv  


In [3]:
# looking for the file sizes

!hdfs dfs -ls /tmp/brazilian-ecommerce

Found 9 items
-rw-r--r--   2 root hadoop    9033957 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_customers_dataset.csv
-rw-r--r--   2 root hadoop   61273883 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_geolocation_dataset.csv
-rw-r--r--   2 root hadoop   15438671 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_order_items_dataset.csv
-rw-r--r--   2 root hadoop    5777138 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_order_payments_dataset.csv
-rw-r--r--   2 root hadoop   14451670 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_order_reviews_dataset.csv
-rw-r--r--   2 root hadoop   17654914 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_orders_dataset.csv
-rw-r--r--   2 root hadoop    2379446 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_products_dataset.csv
-rw-r--r--   2 root hadoop     174703 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_sellers_dataset.csv
-rw-r--r--   2 root hadoop       2613 2026-04-17 16:28 /tmp/brazilian-ecommerce/product_category_name_translation.c

### Data Ingestion and Setup
1. Downloaded Dataset: Retrieved the Brazilian E-commerce dataset from Kaggle using the Kaggle API within the Hadoop cluster environment.
2. Local Storage and Extraction: Stored the downloaded .zip file in the cluster’s local /tmp directory and extracted the dataset files.
3. HDFS Directory Creation: Created a target directory in HDFS at /tmp/brazilian-ecommerce to store the data for distributed processing.
4. Data Transfer to HDFS: Copied the extracted dataset files from the local filesystem (/tmp) to the HDFS directory, making them accessible for Spark processing.

In [1]:
# Creating the Spark session

from pyspark.sql import SparkSession
import spark

spark = SparkSession.builder \
        .appName("OlistData") \
        .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 19:04:30 INFO SparkEnv: Registering MapOutputTracker
26/04/17 19:04:30 INFO SparkEnv: Registering BlockManagerMaster
26/04/17 19:04:30 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/17 19:04:30 INFO SparkEnv: Registering OutputCommitCoordinator


In [2]:
spark

In [3]:
!hadoop fs -ls /tmp/brazilian-ecommerce

Found 9 items
-rw-r--r--   2 root hadoop    9033957 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_customers_dataset.csv
-rw-r--r--   2 root hadoop   61273883 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_geolocation_dataset.csv
-rw-r--r--   2 root hadoop   15438671 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_order_items_dataset.csv
-rw-r--r--   2 root hadoop    5777138 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_order_payments_dataset.csv
-rw-r--r--   2 root hadoop   14451670 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_order_reviews_dataset.csv
-rw-r--r--   2 root hadoop   17654914 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_orders_dataset.csv
-rw-r--r--   2 root hadoop    2379446 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_products_dataset.csv
-rw-r--r--   2 root hadoop     174703 2026-04-17 16:28 /tmp/brazilian-ecommerce/olist_sellers_dataset.csv
-rw-r--r--   2 root hadoop       2613 2026-04-17 16:28 /tmp/brazilian-ecommerce/product_category_name_translation.c

### Reading all the data files

In [4]:
# to simplfy I am writing thhe hdfs_path and calling the files in that folder
hdfs_path = '/tmp/brazilian-ecommerce/'

In [5]:
## Reading the customer file
## With inferSchema as true, Spark automatically detects and assigns appropriate column data types based on the data

customers_df = spark.read \
              .format("csv") \
              .option("header", "true") \
              .option("inferSchema", "true") \
              .load(hdfs_path + 'olist_customers_dataset.csv')

In [6]:
customers_df.show(5)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows



In [7]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [8]:
## Reading the olist_orders_dataset file
orders_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_orders_dataset.csv')


## Reading the olist_order_items_dataset file
order_item_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_order_items_dataset.csv')


## Reading the order_payments file
payments_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_order_payments_dataset.csv')


## Reading the olist_order_reviews_dataset file
reviews_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(hdfs_path + 'olist_order_reviews_dataset.csv')


## Reading the olist_products_dataset file
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv', header=True, inferSchema=True)


## Reading the olist_sellers_dataset file
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv',header=True,inferSchema=True)


## Reading the olist_geolocation_dataset file
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv',header=True,inferSchema=True)


## Reading the olist_orders_dataset file
category_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv',header=True,inferSchema=True)

In [9]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



#### Customer Data Exploration

In [10]:
# Looking for the customer data count

print(f'Customers : {customers_df.count()} rows')

Customers : 99441 rows
Orders : 99441 rows


In [11]:
# NULL or Duplicate Values

customers_df.columns

['customer_id',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

In [12]:
from pyspark.sql.functions import * ##col, when, count,

# Check for null counts in fields
customers_df.select([count(when(col(c).isNull(),1)).alias(c) for c in customers_df.columns]).show()

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [14]:
# Duplicate Values

customers_df.groupBy('customer_id').count().filter('count > 1').show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [17]:
# Customer Distribution by state

customers_df.groupBy('customer_state').count().orderBy(col('count').desc()).show()

+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SP|41746|
|            RJ|12852|
|            MG|11635|
|            RS| 5466|
|            PR| 5045|
|            SC| 3637|
|            BA| 3380|
|            DF| 2140|
|            ES| 2033|
|            GO| 2020|
|            PE| 1652|
|            CE| 1336|
|            PA|  975|
|            MT|  907|
|            MA|  747|
|            MS|  715|
|            PB|  536|
|            PI|  495|
|            RN|  485|
|            AL|  413|
+--------------+-----+
only showing top 20 rows



#### Orders and Payments Data Exploration

In [18]:
# Looking for the order data count

print(f'Orders : {orders_df.count()} rows')

Orders : 99441 rows


In [20]:
orders_df.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [21]:
orders_df.columns

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date']

In [23]:
# Order - Order status distribution

orders_df.groupBy('order_status').count().orderBy(col('count').asc()).show()

+------------+-----+
|order_status|count|
+------------+-----+
|    approved|    2|
|     created|    5|
|  processing|  301|
|    invoiced|  314|
| unavailable|  609|
|    canceled|  625|
|     shipped| 1107|
|   delivered|96478|
+------------+-----+



In [24]:
# Oldest and newest orders purchased per order_status

orders_df.groupBy('order_status').agg(min('order_purchase_timestamp').alias('oldest'), max('order_purchase_timestamp').alias('newest')).show()

+------------+-------------------+-------------------+
|order_status|             oldest|             newest|
+------------+-------------------+-------------------+
|   delivered|2016-09-15 12:16:38|2018-08-29 15:00:37|
|    canceled|2016-09-05 00:15:34|2018-10-17 17:30:18|
|     created|2017-11-06 13:12:34|2018-02-09 17:21:04|
|     shipped|2016-09-04 21:15:19|2018-09-03 09:06:57|
|    approved|2017-02-06 20:18:17|2017-04-25 01:25:34|
|  processing|2016-10-05 22:44:13|2018-07-23 18:03:03|
|    invoiced|2016-10-04 13:02:10|2018-08-14 18:45:08|
| unavailable|2016-10-05 14:16:28|2018-08-21 12:21:00|
+------------+-------------------+-------------------+



In [25]:
# Payments Data

payments_df.show(10)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|        99.33|
|a9810da82917af2d9...|                 1| credit_card|                   1|        24.39|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|        65.71|
|ba78997921bbcdc13...|                 1| credit_card|                   8|       107.78|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|       128.45|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|        96.12|
|771ee386b001f0620...|                 1| credit_card|                   1|        81.16|
|3d7239c394a212faa...|                 1| credit_card|                   3|        51.84|
|1f78449c8

In [26]:
payments_df.groupBy('payment_type').count().orderBy('count',ascending=False).show()

+------------+-----+
|payment_type|count|
+------------+-----+
| credit_card|76795|
|      boleto|19784|
|     voucher| 5775|
|  debit_card| 1529|
| not_defined|    3|
+------------+-----+



In [ ]:
## Order Items Data

In [27]:
order_item_df.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [28]:
# Top Selling Products

from pyspark.sql.functions import *

top_products = order_item_df.groupBy('product_id').agg(sum('price').alias('total_sales'))
top_products.orderBy(col('total_sales').desc()).show(10)

+--------------------+------------------+
|          product_id|       total_sales|
+--------------------+------------------+
|bb50f2e236e5eea01...|           63885.0|
|6cdd53843498f9289...| 54730.20000000005|
|d6160fb7873f18409...|48899.340000000004|
|d1c427060a0f73f6b...| 47214.51000000006|
|99a4788cb24856965...|43025.560000000085|
|3dd2a17168ec895c7...| 41082.60000000005|
|25c38557cf793876c...| 38907.32000000001|
|5f504b3a1c75b73d6...|37733.899999999994|
|53b36df67ebb7c415...| 37683.42000000001|
|aca2eb7d00ea1a7b8...| 37608.90000000007|
+--------------------+------------------+
only showing top 10 rows



In [33]:
## Average delivery Time Analysis from order data

delivery_df = orders_df.select('order_id', 'order_purchase_timestamp', 'order_delivered_customer_date')

In [34]:
delivery_df.show(5)

+--------------------+------------------------+-----------------------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|
+--------------------+------------------------+-----------------------------+
|e481f51cbdc54678b...|     2017-10-02 10:56:33|          2017-10-10 21:25:13|
|53cdb2fc8bc7dce0b...|     2018-07-24 20:41:37|          2018-08-07 15:27:45|
|47770eb9100c2d0c4...|     2018-08-08 08:38:49|          2018-08-17 18:06:29|
|949d5b44dbf5de918...|     2017-11-18 19:28:06|          2017-12-02 00:28:42|
|ad21c59c0840e6cb8...|     2018-02-13 21:18:39|          2018-02-16 18:17:02|
+--------------------+------------------------+-----------------------------+
only showing top 5 rows



In [38]:
## from pyspark.sql.functions import *

delivery_detail_df = delivery_df.withColumn('delivery_time', datediff(col('order_delivered_customer_date'), col('order_purchase_timestamp'))) 

delivery_detail_df.orderBy('delivery_time',ascending=False).show()

+--------------------+------------------------+-----------------------------+-------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|delivery_time|
+--------------------+------------------------+-----------------------------+-------------+
|ca07593549f1816d2...|     2017-02-21 23:31:27|          2017-09-19 14:36:39|          210|
|1b3190b2dfa9d789e...|     2018-02-23 14:57:35|          2018-09-19 23:24:07|          208|
|440d0d17af552815d...|     2017-03-07 23:59:51|          2017-09-19 15:12:50|          196|
|2fb597c2f772eca01...|     2017-03-08 18:09:02|          2017-09-19 14:33:17|          195|
|285ab9426d6982034...|     2017-03-08 22:47:40|          2017-09-19 14:00:04|          195|
|0f4519c5f1c541dde...|     2017-03-09 13:26:57|          2017-09-19 14:38:21|          194|
|47b40429ed8cce3ae...|     2018-01-03 09:44:01|          2018-07-13 20:51:31|          191|
|2fe324febf907e3ea...|     2017-03-13 20:17:10|          2017-09-19 17:00:07|   

In [39]:
delivery_detail_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- delivery_time: integer (nullable = true)

